Задача 1: Создание и заполнение таблиц
● Создайте таблицу authors с полями id, first_name и
last_name. Используйте PRIMARY KEY для поля id

In [14]:
from psycopg import connect, Connection, Cursor
from dotenv import load_dotenv # pip install python-dotenv
from os import environ

load_dotenv("../.env")

connection: Connection = connect(environ["POSTGRESQL_CONNECTION_STRING"] + "/15_DB_homework?connect_timeout=10")

cursor: Cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS authors (
    id INTEGER PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50)
);
""")

connection.commit()

Создайте таблицу books с полями id, title, author_id и
publication_year. Используйте PRIMARY KEY для поля id и
FOREIGN KEY для поля author_id, ссылаясь на таблицу
authors

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    id INTEGER PRIMARY KEY,
    title VARCHAR(100) NOT NULL,
    author_id INTEGER REFERENCES authors(id)
);
""")

connection.commit()

Создайте таблицу sales с полями id, book_id и quantity.
Используйте PRIMARY KEY для поля id и FOREIGN KEY для
поля book_id, ссылаясь на таблицу books

In [16]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY,
    book_id INTEGER REFERENCES books(id),
    quantity INTEGER NOT NULL DEFAULT 0
);
""")

connection.commit()

Чатгпт спасибо, разрешаем приколы по типу неполных данных :)

In [17]:
cursor.execute("ALTER TABLE books ALTER COLUMN author_id DROP NOT NULL;")
connection.commit()

cursor.execute("ALTER TABLE sales ALTER COLUMN book_id DROP NOT NULL;")
connection.commit()

Добавьте несколько авторов в таблицу authors \
● Добавьте несколько книг в таблицу books, указывая
авторов из таблицы authors \
● Добавьте записи о продажах книг в таблицу sales

In [18]:
# CHATGPT THANKS FOR DATA AND ON CONFLICT (id) DO NOTHING!!!

cursor.execute("""
INSERT INTO authors (id, first_name, last_name) VALUES
    (1, 'George', 'Orwell'),
    (2, 'Jane', 'Austen'),
    (3, 'Fyodor', 'Dostoevsky'),
    (4, 'Mark', 'Twain')  -- автор без книг
ON CONFLICT (id) DO NOTHING;
""")
connection.commit()

# Добавляем книги
cursor.execute("""
INSERT INTO books (id, title, author_id) VALUES
    (1, '1984', 1),
    (2, 'Pride and Prejudice', 2),
    (3, 'Crime and Punishment', 3),
    (4, 'Anonymous Tales', NULL),  -- книга без автора
    (5, 'Crime and Punishment 2.0', 3), -- моя строчечка
    (6, 'Crime and Punishment 10.0', 3) -- моя строчечка
ON CONFLICT (id) DO NOTHING;
""")
connection.commit()

# Добавляем продажи
cursor.execute("""
INSERT INTO sales (id, book_id, quantity) VALUES
    (1, 1, 5),
    (2, 2, 3),
    (3, 3, 2),
    (4, NULL, 10),  -- продажа без книги
    (5, 5, 3),
    (6, 6, 15)
ON CONFLICT (id) DO NOTHING;
""")
connection.commit()


Задача 2: Использование JOIN \
● Используйте INNER JOIN для получения списка всех книг и
их авторов. \
● Используйте LEFT JOIN для получения списка всех авторов
и их книг (включая авторов, у которых нет книг). \
● Используйте RIGHT JOIN для получения списка всех книг и
их авторов, включая книги, у которых автор не указан

In [19]:
print('------------------------- INNER JOIN:')

cursor.execute("""SELECT authors.first_name, authors.last_name, books.title FROM books
INNER JOIN authors ON books.author_id = authors.id
;""")

print(cursor.fetchall())


print("------------------------- LEFT JOIN:")

cursor.execute("""SELECT authors.first_name, authors.last_name, books.title FROM authors
LEFT JOIN books ON authors.id = books.author_id""")

print(cursor.fetchall())

print("------------------------- RIGHT JOIN:")

cursor.execute("""SELECT authors.first_name, authors.last_name, books.title FROM authors
RIGHT JOIN books ON authors.id = books.author_id""")

print(cursor.fetchall())

------------------------- INNER JOIN:
[('George', 'Orwell', '1984'), ('Harper', 'Lee', 'To Kill a Mockingbird'), ('Jane', 'Austen', 'Pride and Prejudice'), ('Jane', 'Austen', 'Crime and Punishment 2.0'), ('Jane', 'Austen', 'Crime and Punishment 10.0')]
------------------------- LEFT JOIN:
[('George', 'Orwell', '1984'), ('Harper', 'Lee', 'To Kill a Mockingbird'), ('Jane', 'Austen', 'Pride and Prejudice'), ('Jane', 'Austen', 'Crime and Punishment 2.0'), ('Jane', 'Austen', 'Crime and Punishment 10.0'), ('Mark', 'Twain', None)]
------------------------- RIGHT JOIN:
[('George', 'Orwell', '1984'), ('Harper', 'Lee', 'To Kill a Mockingbird'), ('Jane', 'Austen', 'Pride and Prejudice'), (None, None, 'Anonymous Tales'), ('Jane', 'Austen', 'Crime and Punishment 2.0'), ('Jane', 'Austen', 'Crime and Punishment 10.0')]


Задача 3: Множественные JOIN \
Используйте INNER JOIN для связывания таблиц authors,
books и sales, чтобы получить список всех книг, их авторов
и продаж

In [20]:
cursor.execute("""SELECT authors.first_name, authors.last_name, books.title, sales.quantity FROM authors
INNER JOIN books ON authors.id = books.author_id
INNER JOIN sales ON sales.book_id = books.id""")

print(cursor.fetchall())

[('George', 'Orwell', '1984', 5), ('Harper', 'Lee', 'To Kill a Mockingbird', 3), ('Jane', 'Austen', 'Pride and Prejudice', 7), ('Jane', 'Austen', 'Crime and Punishment 2.0', 3), ('Jane', 'Austen', 'Crime and Punishment 10.0', 15)]


Используйте LEFT JOIN для связывания таблиц authors,
books и sales, чтобы получить список всех авторов, их книг
и продаж (включая авторов без книг и книги без продаж)

In [21]:
cursor.execute("""SELECT authors.first_name, authors.last_name, books.title, sales.quantity FROM authors
LEFT JOIN books ON authors.id = books.author_id
LEFT JOIN sales ON sales.book_id = books.id""")

print(cursor.fetchall())

[('George', 'Orwell', '1984', 5), ('Harper', 'Lee', 'To Kill a Mockingbird', 3), ('Jane', 'Austen', 'Pride and Prejudice', 7), ('Jane', 'Austen', 'Crime and Punishment 2.0', 3), ('Jane', 'Austen', 'Crime and Punishment 10.0', 15), ('Mark', 'Twain', None, None)]


Задача 4: Агрегация данных с использованием JOIN \
● Используйте INNER JOIN и функции агрегации для
определения общего количества проданных книг каждого
автора

In [22]:
cursor.execute("""
SELECT DISTINCT name, SUM(quantity) FROM
    (SELECT authors.first_name as name, SUM(sales.quantity) as quantity FROM books
    INNER JOIN authors ON books.author_id = authors.id
    INNER JOIN sales ON sales.book_id = books.id
    GROUP BY authors.first_name, books.title) AS selected_first_name_count
GROUP BY name
""")

print(cursor.fetchall())

[('Jane', Decimal('25')), ('George', Decimal('5')), ('Harper', Decimal('3'))]


Используйте LEFT JOIN и функции агрегации для
определения общего количества проданных книг каждого
автора, включая авторов без продаж

In [23]:
cursor.execute("""
SELECT DISTINCT name, coalesce(SUM(quantity), 0) FROM
    (SELECT authors.first_name as name, SUM(sales.quantity) as quantity FROM books
    RIGHT JOIN authors ON books.author_id = authors.id
    LEFT JOIN sales ON sales.book_id = books.id
    GROUP BY authors.first_name, books.title) AS selected_first_name_count
GROUP BY name
""")

print(cursor.fetchall())

[('Mark', Decimal('0')), ('Jane', Decimal('25')), ('George', Decimal('5')), ('Harper', Decimal('3'))]


Задача 5: Подзапросы и JOIN
● Найдите автора с наибольшим количеством проданных
книг, используя подзапросы и JOIN

In [24]:
cursor.execute("""
SELECT DISTINCT name, quantity FROM
    (SELECT authors.first_name as name, COALESCE(SUM(sales.quantity), 0) as quantity FROM books
    RIGHT JOIN authors ON books.author_id = authors.id
    LEFT JOIN sales ON sales.book_id = books.id
    GROUP BY authors.first_name, books.title) AS selected_first_name_count
GROUP BY name, quantity
ORDER BY quantity DESC, name ASC
LIMIT 1
""")

print(cursor.fetchall())

[('Jane', 15)]


Найдите книги, которые были проданы в количестве,
превышающем среднее количество продаж всех книг,
используя подзапросы и JOIN

In [25]:
cursor.execute("""
WITH selected_first_name_quantity AS (
    SELECT
        authors.first_name AS name,
        COALESCE(SUM(sales.quantity), 0) AS quantity
    FROM books
    RIGHT JOIN authors ON books.author_id = authors.id
    LEFT JOIN sales ON sales.book_id = books.id
    GROUP BY authors.first_name
),
avg_quantity_cte AS (
    SELECT AVG(quantity) AS avg_quantity FROM selected_first_name_quantity
)
SELECT
    selected_first_name_quantity.name,
    selected_first_name_quantity.quantity
FROM selected_first_name_quantity, avg_quantity_cte
WHERE selected_first_name_quantity.quantity > avg_quantity_cte.avg_quantity
""")

print(cursor.fetchall())

'''MY NOT WORKING CODE:
cursor.execute("""
SELECT selected_first_name_quantity.name, selected_first_name_quantity.quantity, avg_quantity FROM
    (SELECT AVG(quantity) FROM
        (SELECT authors.first_name as name, COALESCE(SUM(sales.quantity), 0) as quantity FROM books
        RIGHT JOIN authors ON books.author_id = authors.id
        LEFT JOIN sales ON sales.book_id = books.id
        GROUP BY authors.first_name, books.title) AS selected_first_name_quantity
    ) AS avg_quantity
""")

print(cursor.fetchall())'''

[('Jane', 25)]


'MY NOT WORKING CODE:\ncursor.execute("""\nSELECT selected_first_name_quantity.name, selected_first_name_quantity.quantity, avg_quantity FROM\n    (SELECT AVG(quantity) FROM\n        (SELECT authors.first_name as name, COALESCE(SUM(sales.quantity), 0) as quantity FROM books\n        RIGHT JOIN authors ON books.author_id = authors.id\n        LEFT JOIN sales ON sales.book_id = books.id\n        GROUP BY authors.first_name, books.title) AS selected_first_name_quantity\n    ) AS avg_quantity\n""")\n\nprint(cursor.fetchall())'